In [5]:
import sys
import torch

print(f"Python 路径: {sys.executable}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"PyTorch 的 CUDA 版本: {torch.version.cuda}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ GPU不可用，将使用CPU")
print(f"当前使用device: {device}")


Python 路径: /home/administratorlrjcjcjs/yes/envs/cv_gpu/bin/python
PyTorch 版本: 2.14.1+cu130
PyTorch 的 CUDA 版本: 13.0
CUDA 是否可用: True
GPU名称: NVIDIA GeForce RTX 5060
当前使用device: cuda


In [8]:
matrix = torch.ones(256, 256, device=device)
product = matrix @ matrix

print(f"结果所在设备: {product.device}")
print(f"左上角的值: {product[0, 0].item():.0f}")
# 只判断设备类型是cuda，避开 cuda 和 cuda:0 对象对比报错
assert product.device.type == device.type
assert product[0, 0].item() == 256.0


结果所在设备: cuda:0
左上角的值: 256


In [7]:
from torch import nn

# 生成训练数据
x = torch.linspace(-1, 1, 256, device=device).reshape(-1, 1)
y = 3 * x - 2

# 模型放到GPU
model = nn.Linear(1, 1).to(device)
with torch.no_grad():
    model.weight.zero_()
    model.bias.zero_()

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

with torch.no_grad():
    initial_loss = criterion(model(x), y).item()
print(f"训练前初始损失: {initial_loss:.4f}")

for _ in range(40):
    optimizer.zero_grad(set_to_none=True)
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()

# 输出训练结果
print(f"训练结束损失: {loss.item():.4f}")
print(f"学到的权重w: {model.weight.item():.4f} (目标=3)")
print(f"学到的偏置b: {model.bias.item():.4f} (目标=-2)")


训练前初始损失: 7.0235
训练结束损失: 0.0000
学到的权重w: 2.9907 (目标=3)
学到的偏置b: -2.0000 (目标=-2)
